In [ ]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
if "snakemake" not in globals():
        from _helpers_notebooks import mock_snakemake
        snakemake = mock_snakemake(
                "plot_magnitude_pull",
                scenario="default", # default, penalty-sa, penalty-ea, penalty-nwa, penalty-oc
                wacc="regional", # regional, uniform
                cost_year="2050",
                interone="hbi",
                trade_chain="supplyconstraint"
        )

In [ ]:
add_iron_ore_cost = True
ironorecost = 97 * 1.59

In [ ]:
config = snakemake.config

### Get steel demand

In [ ]:
steel_demand = pd.read_csv(snakemake.input.steel_demand, index_col="region")

In [ ]:
steel_demand

### Get costs

In [ ]:
df_all = pd.DataFrame()

# Select input files directly from snakemake (avoids hardcoded paths)
input_files = snakemake.input.supply_curves_interone

for fn in input_files:
    # Extract region from filename pattern: {region}_{process}.csv
    region = fn.split("/")[-1].split("_marginal_cost")[0]

    df = pd.read_csv(fn)
    df["region"] = region

    # Calculate the difference in demand to get the quantity supplied at each cost step
    diff = df["demand [t]"].diff()
    diff[0] = diff[1]  # set first value to second to avoid NaN
    df["demand [t]"] = diff

    df_all = pd.concat([df_all, df], ignore_index=True)

# Remove rows where the lcox [EUR/t] value is NaN (e.g. due to missing data for some regions)
df_all = df_all.dropna(subset=["lcox [EUR/t]"])


if add_iron_ore_cost:
    df_all["lcox [EUR/t]"] += ironorecost

df_all.set_index("region", inplace=True)

In [ ]:
costs = df_all[["lcox [EUR/t]", "product_demand_mt"]]

In [ ]:
costs.loc["Europe"]

#### Get costs at demand level

In [ ]:
def interpolate_cost(region, demand, costs):
    if region not in costs.index:
        return np.nan

    sub = costs.loc[[region]].sort_values("product_demand_mt")
    demands = sub["product_demand_mt"].values
    lcox = sub["lcox [EUR/t]"].values

    # exact match
    if demand in demands:
        return lcox[demands == demand].mean()

    # below smallest or above largest -> take the closest edge value
    if demand <= demands[0]:
        return lcox[0]
    if demand >= demands[-1]:
        return lcox[-1]

    # find bracketing points
    idx_upper = np.searchsorted(demands, demand)
    idx_lower = idx_upper - 1

    return (lcox[idx_lower] + lcox[idx_upper]) / 2

In [ ]:
steel_demand["lcox_EUR_per_t"] = steel_demand.apply(
    lambda row: interpolate_cost(row.name, row["SteelDemand_DRI_Mt"], costs),
    axis=1
)
steel_cost_demand = steel_demand

In [ ]:
steel_cost_demand

### Steel production capacities

In [ ]:
steel_production = pd.read_csv(snakemake.input.steel_production)

In [ ]:
# Build ISO3 -> region mapping from config
regions = config["regions"]
iso3_to_region = {code: region for region, codes in regions.items() for code in codes}

# Map each country row to its region
steel_production_new = steel_production.copy()
steel_production_new["region"] = steel_production_new["iso3"].map(
    lambda x: iso3_to_region.get(x, "Other")
)

# Sum crude steel production and population per region, then compute per-capita
steel_production_by_region = (
    steel_production_new
    .groupby("region")[["crude_steel_production_t", "population"]]
    .sum()
)
steel_production_by_region["SteelProductionMt"] = (
    steel_production_by_region["crude_steel_production_t"] / 1e6
)
steel_production_by_region["SteelProductionMtPerCapita"] = (
    steel_production_by_region["crude_steel_production_t"]
    / steel_production_by_region["population"]
)
steel_production_by_region = steel_production_by_region.sort_values(
    "SteelProductionMt", ascending=False
)

steel_production_by_region[["SteelProductionMt", "SteelProductionMtPerCapita"]]


## Plot

In [ ]:
import seaborn as sns

# --- merge production capacity + cost/demand on region ---
plot_df = steel_cost_demand.join(
    steel_production_by_region[["SteelProductionMtPerCapita"]], how="inner"
)
plot_df = plot_df.reset_index()  # region: index -> column

nice_names = config.get("region_nice_names", {})
plot_df["label"] = plot_df["region"].map(lambda r: nice_names.get(r, r))

# --- quadrant thresholds (medians) ---
x_mid = plot_df["SteelProductionMtPerCapita"].median()
y_mid = plot_df["lcox_EUR_per_t"].median()

# --- bubble sizes: area ∝ DRI demand ---
max_bubble = 2500
demand_max = plot_df["SteelDemand_DRI_Mt"].max()
plot_df["bubble"] = (plot_df["SteelDemand_DRI_Mt"] / demand_max * max_bubble).clip(lower=30)

# --- colour palette (one colour per region) ---
palette = sns.color_palette("tab20", n_colors=len(plot_df))

fig, ax = plt.subplots(figsize=(8, 6))

for i, (_, row) in enumerate(plot_df.iterrows()):
    ax.scatter(
        row["SteelProductionMtPerCapita"],
        row["lcox_EUR_per_t"],
        s=row["bubble"],
        color=palette[i],
        alpha=0.75,
        edgecolors="white",
        linewidths=0.8,
        zorder=3,
    )
    ax.annotate(
        row["label"],
        xy=(row["SteelProductionMtPerCapita"], row["lcox_EUR_per_t"]),
        xytext=(0, 7),
        textcoords="offset points",
        fontsize=8,
        ha="center",
        va="bottom",
    )

# --- quadrant dividers ---
ax.axvline(x_mid, color="grey", linestyle="--", linewidth=1.0, alpha=0.6, zorder=2)
ax.axhline(y_mid, color="grey", linestyle="--", linewidth=1.0, alpha=0.6, zorder=2)

# --- quadrant text annotations ---
xl, xr = ax.get_xlim()
yb, yt = ax.get_ylim()
px = (xr - xl) * 0.02
py = (yt - yb) * 0.02

quadrant_labels = [
    (xl + px, yt - py, "top",    "left",  "high renewables pull\n+ small incumbent industry"),
    (xr - px, yt - py, "top",    "right", "high renewables pull\n+ large incumbent steel industry"),
    (xl + px, yb + py, "bottom", "left",  "low production costs\n+ small incumbent industry"),
    (xr - px, yb + py, "bottom", "right", "low renewables pull\n+ large steel industry"),
]
for xq, yq, va, ha, txt in quadrant_labels:
    ax.text(xq, yq, txt, fontsize=8, va=va, ha=ha,
            color="dimgrey", style="italic", alpha=0.8, zorder=2)

# --- axis labels (no title) ---
ax.set_xlabel("Today's Steel Production Capacity (t / person)", fontsize=12)
ax.set_ylabel("Domestic HBI Costs (€/t)", fontsize=12)

# --- bubble-size legend with clean fixed values ---
for mt_val in [10, 100]:
    s = (mt_val / demand_max) * max_bubble
    ax.scatter([], [], s=s, color="grey", alpha=0.5,
               label=f"{mt_val} Mt")
ax.legend(title="DRI Steel Demand", loc="right", frameon=True, fontsize=9, title_fontsize=9)

plt.tight_layout()
plt.savefig(snakemake.output.magnitude_pull_png, dpi=150, bbox_inches="tight")
plt.savefig(snakemake.output.magnitude_pull, bbox_inches="tight")
plt.show()
